# Tutorial: SD-dMFA Model Cookbook: Read, Calibrate, Run, Inspect, Analyze, Plot

This notebook is an end-to-end operational cookbook for the `v5.0` SD-dMFA model.


## Audience, Prerequisites, and Outcomes

**Audience**
- Model developers and analysts working on this repository.

**Prerequisites**
- You can run Python 3.11+ in this repo.
- Dependencies are installed (`pip install -e ".[dev]"`).
- You are comfortable with shell commands and pandas.

**By the end you can**
1. Read and inspect run/scenario configuration.
2. Validate config and exogenous input surfaces.
3. Run baseline and scenario variants (single and batch).
4. Inspect outputs and convergence diagnostics.
5. Compare scenarios and plot indicators.
6. Run calibration and promote/restore patches.
7. Audit realism gates and interpret failures.


## Outline

1. Notebook controls and environment checks.
2. Read and inspect model configuration.
3. Validate config and exogenous inputs.
4. Run baseline/scenario workflows.
5. Inspect outputs and convergence.
6. Analyze scenario deltas and KPIs.
7. Generate standard plot packages.
8. Calibrate and manage config patch lifecycle.
9. Run realism audit and triage.
10. Exercises + extension ideas.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from typing import Iterable

import pandas as pd
import matplotlib.pyplot as plt

try:
    from crm_model.common.io import load_run_config
except Exception:
    load_run_config = None

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)


In [ ]:
# ---- Notebook runtime controls ----

# If True, commands are printed but not executed.
DRY_RUN = False

# Heavy actions can take time; keep them opt-in.
RUN_HEAVY = False
RUN_CALIBRATION = False
RUN_PLOTS = False
RUN_AUDIT = False

# Choose run config here.
CONFIG = "configs/runs/mvp.yml"

# Optional: choose a scenario variant to run in single-run examples.
EXAMPLE_VARIANT = "baseline"


In [ ]:
# ---- Utilities ----

def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "configs").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError("Could not locate repo root from current working directory.")


def sh(cmd: str, *, cwd: Path, check: bool = True) -> subprocess.CompletedProcess | None:
    print(f"$ {cmd}")
    if DRY_RUN:
        return None
    cp = subprocess.run(cmd, cwd=str(cwd), shell=True, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout)
    if cp.stderr.strip():
        print(cp.stderr)
    if check and cp.returncode != 0:
        raise RuntimeError(f"Command failed with code {cp.returncode}: {cmd}")
    return cp


def latest_dir(base: Path) -> Path | None:
    if not base.exists():
        return None
    cands = [p for p in base.iterdir() if p.is_dir() and p.name != "_archive"]
    return sorted(cands)[-1] if cands else None


def load_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


In [ ]:
REPO = find_repo_root()
CONFIG_PATH = (REPO / CONFIG).resolve()
CONFIG_STEM = CONFIG_PATH.stem

print("Repo:", REPO)
print("Config:", CONFIG_PATH)
print("Config stem:", CONFIG_STEM)


## 1) Read and Inspect the Model Configuration

This step confirms dimensions, time windows, and available scenario variants.


In [ ]:
if load_run_config is None:
    raise RuntimeError("crm_model package is not importable. Run `pip install -e \".[dev]\"` first.")

cfg = load_run_config(CONFIG_PATH)
print("Run name:", cfg.name)
print("Time start/end:", cfg.time.start_year, cfg.time.end_year)
print("Reporting start:", cfg.time.report_start_year)
print("Materials:", [m.name for m in cfg.dimensions.materials])
print("Regions:", list(cfg.dimensions.regions))
print("Variants:", len(cfg.variants))
for v in cfg.variants.keys():
    print(" -", v)


In [ ]:
# Quick view of core behavioral blocks from run config

raw = json.loads(CONFIG_PATH.read_text())
for key in [
    "sd_parameters",
    "mfa_parameters",
    "strategy",
    "transition_policy",
    "demand_transformation",
    "scenario_profiles",
]:
    print(f"\n[{key}] present:", key in raw)


## 2) Validate Config and Exogenous Inputs

Run lint and exogenous validation before any scenario execution.


In [ ]:
_ = sh("python scripts/validation/lint_run_configs.py", cwd=REPO)
_ = sh(f"python scripts/validation/validate_exogenous_inputs.py --config {CONFIG}", cwd=REPO)


## 3) Run Workflows

### 3A. Single-variant runs (reporting/calibration)


In [ ]:
_ = sh(
    f"python scripts/run_one.py --config {CONFIG} --variant {EXAMPLE_VARIANT} --phase reporting --save-csv",
    cwd=REPO,
)

_ = sh(
    f"python scripts/run_one.py --config {CONFIG} --variant {EXAMPLE_VARIANT} --phase calibration --save-csv",
    cwd=REPO,
)


### 3B. Batch runs

Use this for all variants or a selected subset.


In [ ]:
# All variants + auto comparison package
if RUN_HEAVY:
    _ = sh(f"python scripts/run_batch.py --config {CONFIG} --phase reporting --save-csv --compare", cwd=REPO)
else:
    print("Skipped heavy batch run. Set RUN_HEAVY=True to execute.")

# Example: selected variants only
selected = "baseline,demand_surge,circularity_push"
if RUN_HEAVY:
    _ = sh(
        f"python scripts/run_batch.py --config {CONFIG} --phase reporting --variants {selected} --save-csv --compare",
        cwd=REPO,
    )


## 4) Inspect Output Artifacts

Run outputs are stored as:
`outputs/runs/<config_stem>/<variant>/<timestamp>/`.


In [ ]:
run_root = REPO / "outputs" / "runs" / CONFIG_STEM / EXAMPLE_VARIANT
latest = latest_dir(run_root)
print("Latest run dir:", latest)

if latest:
    summary_df = load_csv_if_exists(latest / "summary.csv")
    scalar_df = load_csv_if_exists(latest / "indicators" / "scalar_metrics.csv")
    ts_df = load_csv_if_exists(latest / "indicators" / "timeseries.csv")
    conv_df = load_csv_if_exists(latest / "indicators" / "coupling_convergence_iteration.csv")

    print("summary rows:", len(summary_df))
    print("scalar rows:", len(scalar_df))
    print("timeseries rows:", len(ts_df))
    print("convergence rows:", len(conv_df))


In [ ]:
# Slice-level convergence and stress diagnostics
if latest and not summary_df.empty:
    cols = [
        "material", "region", "iterations", "coupling_converged",
        "coupling_convergence_metric", "final_stress_multiplier",
        "final_bottleneck_pressure_mean", "final_collection_rate_mean",
    ]
    show = [c for c in cols if c in summary_df.columns]
    display(summary_df[show].sort_values(["material", "region"]).reset_index(drop=True))


## 5) Analyze Scenario Comparison Tables

Build and inspect comparison package (`summary_comparison`, `delta_vs_baseline`, `scenario_kpis`).


In [ ]:
_ = sh(f"python scripts/analysis/compare_scenarios.py --config {CONFIG}", cwd=REPO)

cmp_root = REPO / "outputs" / "analysis" / "scenario_comparison" / CONFIG_STEM / "latest"
summary_cmp = load_csv_if_exists(cmp_root / "summary_comparison.csv")
delta_cmp = load_csv_if_exists(cmp_root / "delta_vs_baseline.csv")
kpi_cmp = load_csv_if_exists(cmp_root / "scenario_kpis.csv")

print("Comparison folder:", cmp_root)
print("summary_comparison rows:", len(summary_cmp))
print("delta_vs_baseline rows:", len(delta_cmp))
print("scenario_kpis rows:", len(kpi_cmp))


In [ ]:
# KPI ranking (lower avg_final_stress_multiplier is generally better)
if not kpi_cmp.empty:
    display(kpi_cmp.sort_values("avg_final_stress_multiplier").reset_index(drop=True))


In [ ]:
# Example custom analytical cut: service stress deltas by scenario/material/region
if not delta_cmp.empty and "delta_service_stress" in delta_cmp.columns:
    cut = (
        delta_cmp.groupby(["variant", "material", "region"], as_index=False)["delta_service_stress"]
        .mean()
        .sort_values("delta_service_stress")
    )
    display(cut.head(20))


## 6) Plotting Workflows

This repo has script-driven plot packages for reproducible reporting.


In [ ]:
if RUN_PLOTS:
    _ = sh(f"python scripts/analysis/plots/plot_scenario_subset_panels.py --config {CONFIG}", cwd=REPO)

    # Optional: build end-use source + include end-use detail panels
    _ = sh(
        f"python scripts/analysis/plots/plot_stock_in_use_by_end_use_region_scenarios.py --config {CONFIG}",
        cwd=REPO,
    )
    end_use_files = sorted((REPO / "outputs" / "analysis" / "stock_in_use_by_end_use_region_scenarios").glob("**/stock_in_use_by_end_use_region_scenario.csv"))
    if end_use_files:
        end_use_source = end_use_files[-1]
        _ = sh(
            f"python scripts/analysis/plots/plot_scenario_subset_panels.py --config {CONFIG} --end-use-source {end_use_source}",
            cwd=REPO,
        )
else:
    print("Skipped plotting scripts. Set RUN_PLOTS=True to execute.")


In [ ]:
# Quick inline plot from comparison package: scenario KPI profile
if not kpi_cmp.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    k = kpi_cmp.sort_values("avg_final_stress_multiplier").reset_index(drop=True)
    ax.bar(k["variant"], k["avg_final_stress_multiplier"])
    ax.set_ylabel("avg_final_stress_multiplier")
    ax.set_title("Scenario KPI ranking by average stress multiplier")
    ax.tick_params(axis="x", rotation=70)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()


## 7) Calibration Cookbook

This section shows the baseline calibration lifecycle:
1. run optimizer,
2. promote selected patch,
3. optionally restore previous baseline.


In [ ]:
if RUN_CALIBRATION:
    _ = sh(
        f"python scripts/calibration/calibrate_model.py --config {CONFIG} --calibration-spec configs/calibration.yml",
        cwd=REPO,
    )

    patch_files = sorted((REPO / "outputs" / "runs" / "calibration").glob("**/best_config_patch.yml"))
    if not patch_files:
        raise RuntimeError("No best_config_patch.yml found after calibration run.")
    latest_patch = patch_files[-1]
    print("Promoting patch:", latest_patch)
    _ = sh(
        f"python scripts/calibration/calibration_cycle.py promote --config {CONFIG} --patch {latest_patch}",
        cwd=REPO,
    )
else:
    print("Skipped calibration run/promote. Set RUN_CALIBRATION=True to execute.")


In [ ]:
# Optional rollback helper (prints latest snapshot path)
snaps = sorted((REPO / "outputs" / "runs" / "calibration" / "cycle").glob("**/baseline_before.yml"))
if snaps:
    latest_snapshot = snaps[-1]
    print("Latest rollback snapshot:", latest_snapshot)
    print("Rollback command:")
    print(f"python scripts/calibration/calibration_cycle.py restore --config {CONFIG} --snapshot {latest_snapshot}")
else:
    print("No rollback snapshot found yet.")


## 8) Realism Audit and Diagnostics

Use the realism audit to evaluate gate outcomes and identify failing dynamics.


In [ ]:
if RUN_AUDIT:
    _ = sh(
        "python scripts/analysis/audit_scenario_realism.py --config configs/runs/mvp.yml --config configs/runs/r-strategies.yml",
        cwd=REPO,
        check=False,
    )
else:
    print("Skipped audit execution. Set RUN_AUDIT=True to execute.")

audit_csv = REPO / "outputs" / "analysis" / "scenario_realism" / "latest" / "realism_audit.csv"
audit_df = load_csv_if_exists(audit_csv)
if not audit_df.empty:
    display(audit_df)
    print("Failing gates:")
    display(audit_df[audit_df["passed"] == False])


## 9) Observed vs Modeled Comparison Plots

Useful for calibration diagnostics and communication.


In [ ]:
# Example invocation (kept as explicit command to avoid accidental long runs)
cmd = (
    "python scripts/analysis/plots/plot_observed_vs_model.py "
    f"--config {CONFIG} --phase full --baseline-variant baseline --calibrated-variant calibrated"
)
print(cmd)
if RUN_PLOTS:
    _ = sh(cmd, cwd=REPO)


## 10) Scenario Ramp/Profile Authoring Utility

Expand reporting-window profile CSVs into full-horizon YAML payloads.


In [ ]:
cmd = (
    f"python scripts/scenarios/build_reporting_timeseries_profiles.py --config {CONFIG} "
    "--profile data/ramp_profiles/r_strategies/r36_profiles.csv "
    "--profile data/ramp_profiles/r_strategies/r79_profiles.csv"
)
print(cmd)
if RUN_HEAVY:
    _ = sh(cmd, cwd=REPO)


## Common Pitfalls and Fixes

- **Interactive shell strict mode issue (`RPROMPT` unset)**: in VS Code zsh, use `set -eo pipefail` instead of `set -euo pipefail`.
- **No outputs found**: ensure you ran `--save-csv` and confirm `CONFIG` stem matches output folder.
- **Variant not found**: list variants from `load_run_config(CONFIG).variants.keys()`.
- **Audit exits with code 1**: expected when one or more realism gates fail; inspect `realism_audit.csv`.
- **Long runtime**: keep `RUN_HEAVY=False` while iterating on notebook logic.


## Exercises

1. Switch `CONFIG` from `mvp.yml` to `r-strategies.yml`, rerun comparison and list top 3 variants by lowest `avg_final_stress_multiplier`.
2. Choose one variant and inspect `coupling_convergence_iteration.csv`; identify which signal dominates convergence metric.
3. Run realism audit and propose one config-level intervention for each failing gate.


In [ ]:
# Exercise answer scaffold

# Example scaffold for Exercise 1:
# CONFIG = "configs/runs/r-strategies.yml"
# ... rerun comparison cell ...
# display(kpi_cmp.sort_values("avg_final_stress_multiplier").head(3))

pass
